# Math Photo Solver — Обучение в Google Colab

1. Клонирует репозиторий
2. Устанавливает зависимости
3. Генерирует датасет символов (80% train / 20% val)
4. Обучает классификатор ResNet-18
5. Оценивает точность модели
6. Сохраняет веса на Google Drive
7. Тестирует солвер напрямую и через OCR
8. Запускает веб-сайт через ngrok

> **Перед запуском**: `Среда выполнения → Сменить тип среды выполнения → GPU (T4) → Сохранить`

## Шаг 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/math_solver_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Google Drive подключён:', DRIVE_SAVE_DIR)

## Шаг 2. Клонирование репозитория

In [ ]:
REPO_URL = 'https://github.com/sergey2321/sergey2321.git'
!git clone {REPO_URL} /content/math-solver
%cd /content/math-solver
!git checkout claude/ale-yP87K
print('Готово.')

## Шаг 3. Установка зависимостей

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok
print('Зависимости установлены.')

## Шаг 4. Проверка GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ВНИМАНИЕ: GPU не найден. Обучение будет медленным.')

## Шаг 5. Генерация датасета символов (80/20)

In [ ]:
DATASET_COUNT = 10000
DATASET_DIR   = 'dataset/symbols'
!python -m dataset.generator.generate_handwritten \
    --count {DATASET_COUNT} --out {DATASET_DIR} --seed 42
import os
print(f'Train: {len(os.listdir(DATASET_DIR+"/train"))}  |  Val: {len(os.listdir(DATASET_DIR+"/val"))}')

## Шаг 6. Обучение модели

In [ ]:
EPOCHS     = 25
BATCH_SIZE = 128
LR         = 1e-3
MODEL_OUT  = 'backend/models/symbol_clf.pth'
!python -m training.train_ocr \
    --data {DATASET_DIR} --epochs {EPOCHS} \
    --batch {BATCH_SIZE} --lr {LR} --out {MODEL_OUT}

## Шаг 7. Оценка точности модели

In [ ]:
!python -m training.evaluate --data {DATASET_DIR} --model {MODEL_OUT}

## Шаг 8. Сохранение модели на Google Drive

In [ ]:
import shutil
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_{ts}.pth')
shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_latest.pth')
print(f'Модель сохранена: {DRIVE_SAVE_DIR}/symbol_clf_latest.pth')

## Шаг 9. Скачивание модели на локальный компьютер

**Способ 1** — раскомментируй ячейку ниже.  
**Способ 2** — забери из Google Drive: `math_solver_models/symbol_clf_latest.pth`

In [ ]:
# from google.colab import files
# files.download('backend/models/symbol_clf.pth')

---
## Шаг 10А. Тест солвера НАПРЯМУЮ (без OCR)

Генерируем картинки, но решаем по исходному выражению — без OCR.  
Это правильный способ тестирования математического движка.  
OCR нужен только для **реальных фотографий**, а не для сгенерированных изображений — там выражение уже известно.

In [ ]:
import sys, os
sys.path.insert(0, '/content/math-solver')

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from backend.solver.solver import solve
from backend.utils.image_generator import render_expression_image

TEST_EXPRESSIONS = [
    '2 + 2',
    '15 * 3 - 7',
    '2*x + 3 = 7',
    'x**2 - 5*x + 6 = 0',
    'integrate(x**2, x)',
    'diff(x**3 + 2*x, x)',
    '(x + 1)**2',
    '2*x**2 + 3*x - 2 = 0',
]

# Генерируем и показываем картинки
fig, axes = plt.subplots(len(TEST_EXPRESSIONS), 1, figsize=(10, len(TEST_EXPRESSIONS) * 1.3))
for i, expr in enumerate(TEST_EXPRESSIONS):
    png = render_expression_image(expr)
    from PIL import Image as PILImage
    import io
    img = PILImage.open(io.BytesIO(png))
    axes[i].imshow(img)
    axes[i].axis('off')
plt.suptitle('Тестовые задачи', fontsize=13)
plt.tight_layout()
plt.show()

# Решаем НАПРЯМУЮ (выражение известно — OCR не нужен)
print('=' * 65)
print('ТЕСТ СОЛВЕРА (прямое решение, без OCR)')
print('=' * 65)
all_ok = 0
for i, expr in enumerate(TEST_EXPRESSIONS):
    r = solve(expr)
    status = '✓' if r['verified'] else '✗'
    ok_str = 'OK' if not r['error'] else 'ОШИБКА'
    if not r['error']:
        all_ok += 1
    print(f'\n[{i+1}] {expr}')
    print(f'    Ответ  : {r["answer"]}')
    print(f'    Статус : {status} {ok_str}')
    if r['error']:
        print(f'    Ошибка : {r["error"]}')

print(f'\n{"="*65}')
print(f'Решено успешно: {all_ok}/{len(TEST_EXPRESSIONS)}')

---
## Шаг 10Б. Тест OCR на реальном фото

Загрузи своё фото с задачей — OCR его распознает и решит.  
Именно так работает сайт когда ты загружаешь фото.

In [ ]:
from google.colab import files
import io
from PIL import Image
import matplotlib.pyplot as plt

from backend.preprocessing.image_prep import preprocess_image
from backend.ocr.printed import extract_expression
from backend.solver.solver import solve

print('Загрузи фото с математической задачей:')
uploaded = files.upload()

for filename, img_bytes in uploaded.items():
    print(f'\nФайл: {filename}')

    # Показать фото
    img = Image.open(io.BytesIO(img_bytes))
    plt.figure(figsize=(8, 3))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Загруженное фото')
    plt.show()

    # Предобработка
    preprocessed = preprocess_image(img_bytes)

    # OCR
    recognized = extract_expression(preprocessed)
    print(f'Распознано : "{recognized}"')

    if not recognized.strip():
        print('OCR ничего не распознал. Попробуй более чёткое фото.')
        continue

    # Решение
    result = solve(recognized)
    print(f'Ответ      : {result["answer"]}')
    print(f'Проверка   : {"✓ пройдена" if result["verified"] else "✗ не пройдена"}')
    if result['error']:
        print(f'Ошибка     : {result["error"]}')
    print('Шаги:')
    for step in result['steps']:
        print(f'  {step}')

---
## Шаг 11. Запуск веб-сайта с публичным URL (через ngrok)

> **Нужен токен ngrok** — бесплатно на [ngrok.com](https://ngrok.com) → `Your Authtoken`

In [ ]:
NGROK_TOKEN = 'ВСТАВЬ_ТОКЕН_СЮДА'

from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = NGROK_TOKEN

server = subprocess.Popen(
    ['uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/math-solver',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

public_url = ngrok.connect(8000)
print('=' * 55)
print('  Сайт запущен!')
print(f'  Открой: {public_url}')
print('=' * 55)

In [ ]:
ngrok.disconnect(public_url)
server.terminate()
print('Сервер остановлен.')

---
## Бонус: Визуализация датасета

In [ ]:
import json, random, matplotlib.pyplot as plt
from PIL import Image

with open(f'{DATASET_DIR}/metadata.json') as f:
    meta = json.load(f)

samples = random.sample(meta['train'], min(20, len(meta['train'])))
fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for ax, rec in zip(axes.flat, samples):
    img = Image.open(f"{DATASET_DIR}/{rec['file']}")
    ax.imshow(img, cmap='gray')
    ax.set_title(rec['symbol'], fontsize=10)
    ax.axis('off')
plt.suptitle('Примеры символов из датасета', fontsize=14)
plt.tight_layout()
plt.show()